In [1]:
import pandas as pd


In [31]:
sales_df1 = pd.read_excel('Sale Order History.xlsx')
customer_df1 = pd.read_excel("Customer.xlsx")
item1 = pd.read_excel('Item.xlsx')

In [56]:
sales_df=sales_df1.copy()
customer_df=customer_df1.copy()
item=item1.copy()

In [57]:
print(list(sales_df.columns))
print(list(customer_df.columns))
print(list(item.columns))

['SO Internal ID', 'Sales Order Date', 'Sales Order Number', 'Sales Order Status', 'Customer Internal ID ', 'Company Name', 'Item Internal ID ', 'Item Code', 'Base Price', 'Quantity']
['Customer Internal ID ', 'Company Name', 'Customer Type', 'FA Sub Industries']
[' Item Internal ID', 'Item Code', 'Production Status', 'Product Category 1', 'Product Category 2', 'List price']


In [58]:
for data in [sales_df, customer_df, item]:
    data.columns = (
        data.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

In [59]:
print(list(sales_df.columns))
print(list(customer_df.columns))
print(list(item.columns))

['so_internal_id', 'sales_order_date', 'sales_order_number', 'sales_order_status', 'customer_internal_id', 'company_name', 'item_internal_id', 'item_code', 'base_price', 'quantity']
['customer_internal_id', 'company_name', 'customer_type', 'fa_sub_industries']
['item_internal_id', 'item_code', 'production_status', 'product_category_1', 'product_category_2', 'list_price']


simple data cleaning

In [60]:
sales_df["company_name"] = sales_df["company_name"].astype(str).str.strip()
customer_df["company_name"] = customer_df["company_name"].astype(str).str.strip()
item["item_internal_id"].astype(str).str.strip()


sales_df["customer_internal_id"] = sales_df["customer_internal_id"].astype(str).str.strip()
customer_df["customer_internal_id"] = customer_df["customer_internal_id"].astype(str).str.strip()

merge sales and customer table

In [61]:
# Merge on BOTH Customer Name and Internal ID
merged_df = sales_df.merge(
    customer_df,
    on=["company_name", "customer_internal_id"],
    how="left",
    validate="many_to_one"   # Each customer should appear only once in Customer.xlsx
)

# Report rows that didn't find a match
unmatched = merged_df[merged_df.isna().any(axis=1)]

print(f"Total rows: {len(merged_df)}")
print(f"Matched rows: {len(merged_df) - len(unmatched)}")
print(f"Unmatched rows: {len(unmatched)}")

if not unmatched.empty:
    print("\nCustomers with no matching record:")
    print(unmatched[["Company Name", "Customer Internal ID"]].drop_duplicates())

# merged_df now contains all customer information
df = merged_df.drop_duplicates(keep='first')
df["sales_order_number"] = df["sales_order_number"].str.upper()
df['item_internal_id'].astype(str).str.strip()
 

df = df.merge(item, on=['item_internal_id','item_code'], how='left', validate="many_to_one")

Total rows: 272000
Matched rows: 272000
Unmatched rows: 0


C:\Users\yoth99\AppData\Local\Temp\ipykernel_4052\2674828800.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["sales_order_number"] = df["sales_order_number"].str.upper()


look for inconsistent data (only step that remain is the one I found inconsistency)

In [ ]:
# SO Internal ID -> Sales Order Number
id_to_order = (
    df.groupby("so_internal_id")["sales_order_number"]
        .nunique()
)

bad_internal_ids = id_to_order[id_to_order > 1]

print(f"{len(bad_internal_ids)} SO Internal IDs map to multiple Sales Order Numbers.\n")


0 SO Internal IDs map to multiple Sales Order Numbers.



In [70]:
# Sales Order Number -> SO Internal ID
order_to_id = (
    df.groupby("sales_order_number")["so_internal_id"]
        .nunique()
)

bad_orders = order_to_id[order_to_id > 1]

print(f"{len(bad_orders)} Sales Order Numbers map to multiple SO Internal IDs.\n")


40 Sales Order Numbers map to multiple SO Internal IDs.



In [64]:
problem_orders = bad_orders.index

problem_rows = (
    df[df["sales_order_number"].isin(problem_orders)]
    .sort_values(["sales_order_number", "so_internal_id"])
)

print(problem_rows[["so_internal_id","sales_order_number","company_name"]].head(15))

       so_internal_id sales_order_number  company_name
21557         7305035               AMXB  AAJ CO.,LTD.
21558         7305035               AMXB  AAJ CO.,LTD.
21559         7305035               AMXB  AAJ CO.,LTD.
21560         7305036               AMXB  AUN CO.,LTD.
28590         7343876               AQZO  AAB CO.,LTD.
28591         7343876               AQZO  AAB CO.,LTD.
28592         7343877               AQZO  AAB CO.,LTD.
36892         7413710               AWBF  AAC CO.,LTD.
36893         7413711               AWBF  AAG CO.,LTD.
38742         7434676               AXFD  ARK CO.,LTD.
38743         7434680               AXFD  AAC CO.,LTD.
38744         7434680               AXFD  AAC CO.,LTD.
39304         7441447               AXPJ  AAC CO.,LTD.
39307         7441460               AXPJ  AAC CO.,LTD.
39308         7441460               AXPJ  AAC CO.,LTD.


#### Data inconsistency in this case can be,

1. different internal ID, same order number, same company
2. different internal ID, same order number, different company

since there's no clear pattern, I mapped every order to a composite key using order number and customer internal ID
this is used for pair purchasing analysis to a certain extent

In [ ]:
df.head()

,order_key,so_internal_id,sales_order_date,sales_order_number,sales_order_status,customer_internal_id,company_name,item_internal_id,item_code,base_price,quantity,customer_type,fa_sub_industries,production_status,product_category_1,product_category_2,list_price
0,AAAA_5356,7165143,2021-04-01,AAAA,Billed,5356,"AAH CO.,LTD.",8523,CFN,14426.0,1,VIP,Wholesale and Distribution,Discontinue,INV,INV E700 Nagoya,16436.0
1,AAAA_5356,7165143,2021-04-01,AAAA,Billed,5356,"AAH CO.,LTD.",8533,CFY,15926.0,1,VIP,Wholesale and Distribution,Discontinue,INV,INV E700 Nagoya,18161.0
2,AAAA_5356,7165143,2021-04-01,AAAA,Billed,5356,"AAH CO.,LTD.",20142,CGX,2326.0,1,VIP,Wholesale and Distribution,Continue,INV,INV Others,2521.0
3,AAAA_5356,7165143,2021-04-01,AAAA,Billed,5356,"AAH CO.,LTD.",8552,CGL,25626.0,1,VIP,Wholesale and Distribution,Discontinue,INV,INV E700 Nagoya,29316.0
4,AAAB_5356,7165145,2021-04-01,AAAB,Billed,5356,"AAH CO.,LTD.",24548,CXR,15826.0,1,VIP,Wholesale and Distribution,Continue,PLC,PLC FX,18120.0


In [66]:
# Create a composite key using Sales Order Number and Customer Internal ID
df.insert(
    0,  # Insert as the first column
    "order_key",
    (
        df["sales_order_number"].astype(str).str.strip()
        + "_"
        + df["customer_internal_id"].astype(str).str.strip()
    )
)

In [68]:
categorical_cols = [
    "sales_order_status",
    "customer_type",
    "production_status",
    "product_category_1",
    "product_category_2",
    "fa_sub_industries"
]

for col in categorical_cols:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
    )

In [69]:
df.head()

,order_key,so_internal_id,sales_order_date,sales_order_number,sales_order_status,customer_internal_id,company_name,item_internal_id,item_code,base_price,quantity,customer_type,fa_sub_industries,production_status,product_category_1,product_category_2,list_price
0,AAAA_5356,7165143,2021-04-01,AAAA,Billed,5356,"AAH CO.,LTD.",8523,CFN,14426.0,1,VIP,Wholesale and Distribution,Discontinue,INV,INV E700 Nagoya,16436.0
1,AAAA_5356,7165143,2021-04-01,AAAA,Billed,5356,"AAH CO.,LTD.",8533,CFY,15926.0,1,VIP,Wholesale and Distribution,Discontinue,INV,INV E700 Nagoya,18161.0
2,AAAA_5356,7165143,2021-04-01,AAAA,Billed,5356,"AAH CO.,LTD.",20142,CGX,2326.0,1,VIP,Wholesale and Distribution,Continue,INV,INV Others,2521.0
3,AAAA_5356,7165143,2021-04-01,AAAA,Billed,5356,"AAH CO.,LTD.",8552,CGL,25626.0,1,VIP,Wholesale and Distribution,Discontinue,INV,INV E700 Nagoya,29316.0
4,AAAB_5356,7165145,2021-04-01,AAAB,Billed,5356,"AAH CO.,LTD.",24548,CXR,15826.0,1,VIP,Wholesale and Distribution,Continue,PLC,PLC FX,18120.0


In [ ]:
with open("merged_output.csv", "w", newline="", encoding="utf-8") as f:
    df.to_csv(f, index=False)

In [ ]:
head_df = df.head(100)
with open("example_data.csv", "w", newline="", encoding="utf-8") as f:
    head_df.to_csv(f, index=False)